<header>
   <p style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
        Content-Based Collections across Multiple Contexts
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

## Introduction

teradataml supports **multiple connections (contexts) at the same time**. Every call to `create_context()` returns a *context object* (`ctx`).

```python
ctx1 = create_context(host="host1", ...)
ctx2 = create_context(host="host2", ...)   # ctx2 is now the active context
```

Two rules to keep in mind:

1. **The active context is always the one created last** (here `ctx2`). Calls made without a context, for example `DataFrame(...)` or `load_data(...)`, go to the active context.
2. **A DataFrame remembers where it came from.** `ctx1.DataFrame(...)` belongs to `ctx1`.


<b>Notes:</b>
* An **auth token is not a context**. A collection can work with `auth_data` alone, and one context can hold several auth tokens (Basic, JWT, ...). `ctx` decides only where the database-side work runs.
* If `auth_data` was created on a different context than `ctx`, or an input DataFrame belongs to a different context, a clear error is raised.
* If you work with a single context, nothing changes. See `Content_Based.ipynb`.

### What You'll Learn

1. Connecting to two Vantage systems at the same time
2. Creating an auth token and loading data on each context
3. Creating and querying a content-based collection on each context
4. What happens when contexts are mixed
5. Using SQL mode on a chosen context
6. Cleaning up resources

---
## 1. Setup

### Install teradatagenai

In [ ]:
# %pip install teradatagenai --upgrade

### Import Required Modules

In [ ]:
from teradatagenai import Collection, CollectionManager, TeradataAI, load_data, ContentBasedIndex, ColumnInfo
from teradatagenai.vector_store.data_classes import HNSW
from teradatagenai.common.constants import CollectionType
from teradatagenai.common.exceptions import TeradataGenAIException
from teradataml import create_context
from teradatasqlalchemy.types import VARCHAR

### Connect to two Vantage systems

Keep a reference to each context object returned by `create_context()`.

In [ ]:
from getpass import getpass
import os
# Enter the credentials for system 1 when prompted
host1 = getpass("System 1 TD_HOST: ")
username1 = getpass("System 1 TD_USERNAME: ")
password1 = getpass("System 1 TD_PASSWORD: ")
base_url1 = getpass("System 1 base URL (e.g. https://<host>/one-td/): ")
ctx1 = create_context(host=host1, username=username1, password=password1)
print("✓ Connected to system 1")


# base_url1 = getpass("System 1 alternative base URL: ")
# host1 = getpass("System 1 alternative host: ")
# username1 = getpass("System 1 alternative username: ")
# password1 = getpass("System 1 alternative password: ")
# ctx1 = create_context(host=host1, username=username1, password=password1)
# print("✓ Connected to system 1")
# print("Active context is ctx1:", ctx1.is_global_context())

✓ Connected to system 1


In [13]:
# Connection to the second Vantage system. This is now the active context.
base_url2 = getpass("System 2 base URL (e.g. https://<host>/one-td/): ")
host2 = getpass("System 2 host: ")
username2 = getpass("System 2 username: ")
password2 = getpass("System 2 password: ")
ctx2 = create_context(host=host2, username=username2, password=password2)
print("✓ Connected to system 2")
print("Active context is ctx2:", ctx2.is_global_context())

/usr/local/lib/python3.12/dist-packages/teradatasqlalchemy/telemetry/queryband.py:557: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting a global context associated with Teradata Vantage Connection. Use 'context' object to invoke/access the functions/DataFrames related to older context.
  return exposed_func(*args, **kwargs)


✓ Connected to system 2
Active context is ctx2: True


### Set the authentication token on each context

Use `ctx.set_auth_token()` so that each token is created on its own context. Refer to `help(set_auth_token)` for other authentication mechanisms.

In [7]:
auth1 = ctx1.set_auth_token(base_url=base_url1,
                            username=username1,
                            password=password1,
                            auth_mech="BASIC")

auth2 = ctx2.set_auth_token(base_url=base_url2,
                            username=username2,
                            password=password2,
                            auth_mech="BASIC")
print("✓ Auth tokens set")

Authentication token is generated and set for the session.
Authentication token is generated and set for the session.
✓ Auth tokens set


### Load Example Data on each context

Pass `ctx` to `load_data()` and create the DataFrames **through the context** using `ctx.DataFrame(...)`.

<b>Note:</b> After `ctx2` is created, a plain `DataFrame("amazon_reviews_25")` goes to `ctx2`. Always use `ctx1.DataFrame(...)` / `ctx2.DataFrame(...)` when working with multiple contexts.

In [8]:
load_data('byom', 'amazon_reviews_25', ctx=ctx1)
load_data('byom', 'amazon_reviews_25', ctx=ctx2)

reviews_ctx1 = ctx1.DataFrame("amazon_reviews_25")
reviews_ctx2 = ctx2.DataFrame("amazon_reviews_25")

print("reviews_ctx1 belongs to ctx1:", reviews_ctx1.ctx is ctx1)
print("reviews_ctx2 belongs to ctx2:", reviews_ctx2.ctx is ctx2)
reviews_ctx1.head(1)

reviews_ctx1 belongs to ctx1: True
reviews_ctx2 belongs to ctx2: True


rev_id,aid,rev_name,helpful,rev_text,rating,prodsummary,unixrevtime,revtime
A10000012B7CGYKOMPQ4L,000100039X,Adam,"[0, 0]",Spiritually and mentally inspiring! A book that allows you to question your morals and will help you discover who you really are!,5.000,Wonderful!,1355616000,"12 16, 2012"


### Check the collection service on each context

`CollectionManager` uses `auth_data` for the REST calls and the context for the `teradataml` results.

In [9]:
cm1 = CollectionManager(auth_data=auth1, ctx=ctx1)
cm2 = CollectionManager(auth_data=auth2, ctx=ctx2)

print("System 1:", cm1.health(return_type="json"))
print("System 2:", cm2.health(return_type="json"))

System 1: {'status': 'up', 'platform': 'aws', 'commit': 'dad84e6', 'version': '2.4.7'}
System 2: {'status': 'up', 'platform': 'aws', 'commit': '1b1139f', 'version': '2.4.10'}


In [10]:
models_ctx1 = cm1.list_available_models()
models_ctx1

embedding_models:
                        model_id                      model_name provider  status
0   cohere.embed-multilingual-v3              Embed Multilingual   Cohere  ACTIVE
1        cohere.embed-english-v3                   Embed English   Cohere  ACTIVE
2   amazon.titan-embed-text-v2:0        Titan Text Embeddings V2   Amazon  ACTIVE
3    amazon.titan-embed-image-v1  Titan Multimodal Embeddings G1   Amazon  ACTIVE
4  amazon.titan-embed-g1-text-02        Titan Text Embeddings v2   Amazon  ACTIVE
5     amazon.titan-embed-text-v1      Titan Embeddings G1 - Text   Amazon  ACTIVE

chat_models:
                           model_id                    model_name     provider  status
0           qwen.qwen3-next-80b-a3b            Qwen3 Next 80B A3B         Qwen  ACTIVE
1  mistral.mistral-7b-instruct-v0:2           Mistral 7B Instruct   Mistral AI  ACTIVE
2   meta.llama3-1-70b-instruct-v1:0        Llama 3.1 70B Instruct         Meta  ACTIVE
3      meta.llama3-8b-instruct-v1:0           

---
## 2. Initialize Models

Models are not tied to a context. The same `TeradataAI` objects are used for both systems.

In [25]:
hosted_embedding_model = TeradataAI(api_type="td_hosted",
                                    model_name="amazon.titan-embed-text-v2:0")

hosted_chat_model = TeradataAI(api_type="td_hosted",
                               model_name="qwen.qwen3-next-80b-a3b")

print("✓ Embedding and Chat models initialized")

✓ Embedding and Chat models initialized


---
## 3. Create a Content-Based Collection on each context

The index is built from the DataFrame of the **same** context the collection is bound to.

In [ ]:
def build_index(reviews_df):
    """Build a ContentBasedIndex for the amazon reviews DataFrame."""
    rev_id_column = ColumnInfo(name="rev_id", 
                               datatype=VARCHAR(50),
                               description="Review ID - Primary Key",
                               sources=[reviews_df.rev_id])
    
    rev_text_column = ColumnInfo(name="rev_text", 
                                 datatype=VARCHAR(64000),
                                 description="Review text for embeddings",
                                 sources=[reviews_df.rev_text])
    
    prodsummary_column = ColumnInfo(name="prodsummary", 
                                    datatype=VARCHAR(4000),
                                    description="Product summary metadata",
                                    sources=[reviews_df.prodsummary])
    
    return ContentBasedIndex(object_names=reviews_df,
                             key_columns=[rev_id_column],
                             data_columns=[rev_text_column],
                             metadata_columns=[prodsummary_column])

### Create on system 1 using `create()`

`ctx1` is **not** the active context, but the collection works on it because it is bound with `ctx=ctx1`.

In [13]:
collection1 = Collection(name="mc_amazon_reviews", auth_data=auth1, ctx=ctx1)

collection1.create(type=CollectionType.CONTENT_BASED,
                   index=build_index(reviews_ctx1),
                   embedding_model=hosted_embedding_model,
                   chat_model=hosted_chat_model)

print(f"✓ Collection '{collection1.name}' created on system 1")

/root/sm/LCL/lcl-NexusAI/teradatagenai/vector_store/collection.py:776: UserWarning: Collection does not exist or name is not supplied. Create it before proceeding ahead.
  warnings.warn("Collection does not exist or name is not supplied. Create it before proceeding ahead.")


Collection create request is accepted and in progress
✓ Collection 'mc_amazon_reviews' created on system 1


In [14]:
collection1.status()

collection_name,collection_status,retry_after
mc_amazon_reviews,create_started,60


### Create on system 1 with ctx2 using `from_datasets()`

The `from_*()` class methods accept `ctx` as well.

In [ ]:
collection2 = Collection.from_datasets(name="mc_amazon_reviews",
                                       data=reviews_ctx2,
                                       embedding=hosted_embedding_model,
                                       key_columns=ColumnInfo(name="rev_id", datatype=VARCHAR(50),
                                                              sources=[reviews_ctx2.rev_id]),
                                       data_columns=ColumnInfo(name="rev_text", datatype=VARCHAR(64000),
                                                               sources=[reviews_ctx2.rev_text]),
                                       chat_model=hosted_chat_model,
                                       auth_data=auth2,
                                       ctx=ctx2)

print(f"✓ Collection '{collection2.name}' created on system 2")

✓ Collection 'mc_amazon_reviews' created on system 2


In [19]:
collection2.status()

collection_name,collection_status
mc_amazon_reviews,ready


---
## 4. Search on each context

`teradataml` results are created on the context the collection is bound to.

In [20]:
question = 'Positive reviews'

search_ctx1 = collection1.similarity_search(question=question, top_k=5)
search_ctx2 = collection2.similarity_search(question=question, top_k=5)

print("Results of system 1 belong to ctx1:", search_ctx1.similar_objects.ctx is ctx1)
print("Results of system 2 belong to ctx2:", search_ctx2.similar_objects.ctx is ctx2)
search_ctx1

Results of system 1 belong to ctx1: True
Results of system 2 belong to ctx2: True


similar_objects_count:5
similar_objects:
      score    DataBaseName          TableName  TD_ID          rev_id                                         prodsummary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         rev_text  ind

In [21]:
search_ctx2

similar_objects_count:5
similar_objects:
      score    DataBaseName          TableName  TD_ID          rev_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         rev_text  index_label
0  0.145638  td_genai_user2  amazon_reviews

In [26]:
question = 'Are there any reviews saying that the books are inspiring?'

print("System 1:")
print(collection1.ask(question=question, top_k=5, chat_model=hosted_chat_model))

print("\nSystem 2:")
print(collection2.ask(question=question, top_k=5, chat_model=hosted_chat_model))

System 1:


Yes, multiple reviews describe the book as inspiring. One review calls it 'Spiritually and mentally inspiring!' and another states that Gibran's work is 'moving and inspirational,' noting it can be 'far more encouraging and moving than any self-help program.' Another reviewer mentions that the book 'touches my heart... again and again' and calls it a 'treasure,' implying deep emotional and inspirational impact.

System 2:
Yes, several reviews describe the book as inspiring. One review calls it 'Spiritually and mentally inspiring!' and mentions it helps readers discover who they really are. Another describes the work as 'moving and inspirational,' and notes it can be 'far more encouraging and moving than any self-help program.' Another reviewer states, 'I would never get enough of it! it's a treasure,' implying deep inspiration.


In [27]:
# Collection tables and views are read through the bound context too.
embeddings_ctx1 = collection1.get_indexes_embeddings()
print("Embeddings table belongs to ctx1:", embeddings_ctx1.ctx is ctx1)
embeddings_ctx1.head(1)

Embeddings table belongs to ctx1: True


DataBaseName,TableName,TD_ID,rev_id,prodsummary,rev_text,vector_index,vector_index_normalized
td_genai_user2,amazon_reviews_25,6,ARDQ9KNB8K22N,"""""""The Prophet"""" is cool""","Cool book, I really like the quality of the production. Black clothbound with gold embossing and nice paper, looks to be cotton rag. I am proud to include it in my collection. Intrigueing story full of the sort of passion that does not cloud the mind or divert truth but instead is revealing and living. Very readable, the stories are short and highly economic so that one is likely to read for only a minute or two until something insightful is revealed. It is full of timeless truths which are of lasting value to the reader and inform life. The language and delivery is familiar yet of a quality that is penetrating in a similar way as it might be listening to a surmon by Jesus. I give it 5 stars. I considered giving only 4 stars because the format is fairly predictable but I have decided that this quality is actually a strength as one can pick up the book at any point and continue until the end...and then start over for that matter. Excellent for busy people or commuters!","-0.0864093825221062,0.0181577391922474,-0.0125464564189315,-0.0386728085577488,-0.00136973324697465,-0.035244420170784,0.0495871789753437,-0.0255698189139366,-0.0182883739471436,0.0229416973888874,0.0168891660869122,0.0381690748035908,0.0402351655066013,-0.0156703684478998,-0.0146901588886976,0.0122022535651922,0.0365770161151886,-0.0225228443741798,0.00157426425721496,0.0290516018867493,0.0209454018622637,-0.0184325948357582,0.0249547474086285,0.0248384214937687,-0.0503890849649906,-0.0231192894279957,0.0426952689886093,-0.0216293390840292,0.0509619563817978,-0.0036863568238914,0.0861783996224403,0.112024866044521,0.0417271181941032,0.0014091677730903,0.0544672720134258,0.091228149831295,0.00616736942902207,0.0701463744044304,-0.0146156502887607,0.0128389643505216,0.025641268119216,0.02092370018363,-0.00340678147040308,0.0156891141086817,0.0202339980751276,0.00251816445961595,-0.00368856359273195,0.0562230497598648,0.080107718706131,-0.0115667106583714,0.0142218824476004,0.0315294712781906,0.0130411610007286","-0.0864093825221062,0.0181577391922474,-0.0125464564189315,-0.0386728085577488,-0.00136973324697465,-0.035244420170784,0.0495871789753437,-0.0255698189139366,-0.0182883739471436,0.0229416973888874,0.0168891660869122,0.0381690748035908,0.0402351655066013,-0.0156703684478998,-0.0146901588886976,0.0122022535651922,0.0365770161151886,-0.0225228443741798,0.00157426425721496,0.0290516018867493,0.0209454018622637,-0.0184325948357582,0.0249547474086285,0.0248384214937687,-0.0503890849649906,-0.0231192894279957,0.0426952689886093,-0.0216293390840292,0.0509619563817978,-0.0036863568238914,0.0861783996224403,0.112024866044521,0.0417271181941032,0.0014091677730903,0.0544672720134258,0.091228149831295,0.00616736942902207,0.0701463744044304,-0.0146156502887607,0.0128389643505216,0.025641268119216,0.02092370018363,-0.00340678147040308,0.0156891141086817,0.0202339980751276,0.00251816445961595,-0.00368856359273195,0.0562230497598648,0.080107718706131,-0.0115667106583714,0.0142218824476004,0.0315294712781906,0.0130411610007286"


---
## 5. Mixing contexts

Input tables must be on the same system as the collection, and `auth_data` must be created on the same context as `ctx`. Otherwise a clear error is raised before any request is sent.

In [28]:
# DataFrame from ctx2 used with a collection bound to ctx1.
try:
    collection1.update(index=ContentBasedIndex(object_names=reviews_ctx2),
                       alter_operation='ADD',
                       update_style='MINOR')
except TeradataGenAIException as err:
    print(err)

[Teradata][teradatagenai](TDGENAI_2004) 'index.object_names' belongs to a different teradataml context than the one 'Collection' is bound to. Create the DataFrame using the same context, e.g. 'ctx.DataFrame(...)'.


In [29]:
# auth_data created on ctx2 used with ctx=ctx1.
try:
    Collection(name="mc_amazon_reviews", auth_data=auth2, ctx=ctx1)
except TeradataGenAIException as err:
    print(err)

[Teradata][teradatagenai](TDGENAI_2004) 'auth_data' belongs to a different teradataml context than the one 'Collection' is bound to. Create "auth_data" on the same context using 'ctx.set_auth_token()'.


---
## 6. SQL mode on a chosen context (optional)

With `use_sql=True`, the collection operations run through `TD_SYSAI` stored procedures on the connection of the given context.

<b>Note:</b> Requires the `TD_SYSAI` stored procedures on that system.

In [30]:
collection1_sql = Collection(name="mc_amazon_reviews", use_sql=True, ctx=ctx1)
collection1_sql.status()

Collection 'mc_amazon_reviews' initialized in SQL mode.


collection_status,status,message
ready,0,Collection status retrieved successfully.


---
## 7. Cleanup

### Destroy the Collections

In [ ]:
collection1.destroy()
collection2.destroy()
print("✓ Collections destroyed")

### Remove Database Connections

In [ ]:
ctx2.remove_context()
ctx1.remove_context()
print("✓ Database connections removed")

---
## Summary

In this notebook, you learned how to:

✅ Connect to two Vantage systems at the same time  
✅ Set an auth token and load data on each context  
✅ Create and query content-based collections bound to a context using `ctx`  
✅ Get `teradataml` results on the context the collection is bound to  
✅ Recognise the errors raised when contexts are mixed  
✅ Use SQL mode on a chosen context  